0. Imports

In [2]:
import pandas as pd
from sqlalchemy import create_engine

1. Exploração rápida de todas as tabelas

Explorando as tabelas para mostrar:

- Quantidade de linhas/colunas;
- Tipos de dados;
- Valores nulos.

In [5]:
tables = {
    'customers': 'olist_customers_dataset.csv',
    'geolocation': 'olist_geolocation_dataset.csv',
    'order_items': 'olist_order_items_dataset.csv',
    'order_payments': 'olist_order_payments_dataset.csv',
    'order_reviews': 'olist_order_reviews_dataset.csv',
    'orders': 'olist_orders_dataset.csv',
    'products': 'olist_products_dataset.csv',
    'sellers': 'olist_sellers_dataset.csv',
    'category_translation': 'product_category_name_translation.csv'
}

dataframes = {}
for name, archive in tables.items():
    df = pd.read_csv(f'../data/raw/{archive}')
    #Salvando as tabelas em um dicionário com 'chave: valor' sendo 'nome da tabela: tabela'
    dataframes[name] = df
    print(f'\n=== {name} ===\nLINHAS/COLUNAS + TIPOS DE DADOS:')
    df.info()
    print(f'PORCENTAGEM DE VALORES NULOS:\n{df.isnull().sum() / len(df) * 100}')


=== customers ===
LINHAS/COLUNAS + TIPOS DE DADOS:
<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   customer_id               99441 non-null  str  
 1   customer_unique_id        99441 non-null  str  
 2   customer_zip_code_prefix  99441 non-null  int64
 3   customer_city             99441 non-null  str  
 4   customer_state            99441 non-null  str  
dtypes: int64(1), str(4)
memory usage: 3.8 MB
PORCENTAGEM DE VALORES NULOS:
customer_id                 0.0
customer_unique_id          0.0
customer_zip_code_prefix    0.0
customer_city               0.0
customer_state              0.0
dtype: float64

=== geolocation ===
LINHAS/COLUNAS + TIPOS DE DADOS:
<class 'pandas.DataFrame'>
RangeIndex: 1000163 entries, 0 to 1000162
Data columns (total 5 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------   

- Lista de todas as relações entre as tabelas.

In [6]:
#Criando um dicionário onde a chave é o nome da tabela e o valor é o conjunto de colunas dela
#Usamos set() para facilitar a comparação de colunas entre tabelas
columns = {name: set(df.columns) for name, df in dataframes.items()}

#Extraindo só os nomes das tabelas em uma lista
#Necessário para iterar por índice nos loops abaixo
tables = list(columns.keys())

#Loop externo: percorre cada tabela como referência
for i in range(len(tables)):
    '''Loop interno: percorre as tabelas seguintes para comparar com a tabela de referência
       j começa em i+1 para não comparar a mesma tabela consigo mesma
       e não repetir pares já comparados (ex: A↔B não será comparado novamente como B↔A)'''
    for j in range(i+1, len(tables)):
        #Pegando os nomes das duas tabelas do par atual
        t1, t2 = tables[i], tables[j]
        '''Encontrando colunas em comum entre as duas tabelas
           O operador & retorna a interseção dos dois conjuntos
           Colunas em comum são as possíveis chaves de relacionamento'''
        common_columns = columns[t1] & columns[t2]
        '''Só imprime se houver pelo menos uma coluna em comum
           Pares sem colunas em comum não têm relacionamento direto'''
        if common_columns:
            print(f"{t1} ↔ {t2}: {common_columns}")

customers ↔ orders: {'customer_id'}
order_items ↔ order_payments: {'order_id'}
order_items ↔ order_reviews: {'order_id'}
order_items ↔ orders: {'order_id'}
order_items ↔ products: {'product_id'}
order_items ↔ sellers: {'seller_id'}
order_payments ↔ order_reviews: {'order_id'}
order_payments ↔ orders: {'order_id'}
order_reviews ↔ orders: {'order_id'}
products ↔ category_translation: {'product_category_name'}


A partir desse resultado, abtrai-se que:

- olist_customers_dataset -> orders (FK: customer_id);
- olist_orders_dataset -> olist_order_items_dataset (FK: order_id);
- olist_orders_dataset -> olist_order_payments_dataset (FK: order_id);
- olist_orders_dataset -> olist_order_reviews_dataset (FK: order_id);
- olist_products_dataset -> olist_order_items_dataset (FK: product_id);
- olist_sellers_dataset -> olist_order_items_dataset (FK: seller_id);
- product_category_name_translation -> olist_products_dataset (FK: product_category_name).

2. Extração, Transformação e Carga (ETL)